# 1D loop extrusion

Cohesin (LEF) dynamics on a 1D lattice with CTCF boundaries — no polymer, no OpenMM.
Fast enough to explore parameters interactively before committing to a 3D run
(`run_sim3D.ipynb`), and the output is the raw LEF trajectory.

Everything comes from `extrusion.py`.

In [12]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from polysim import extrusion, OUTPUTS

# the first import compiles LEF_Dynamics.pyx via pyximport -- a few seconds, once
print("ready")

ready


## Parameters

`CTCF_LEFT` blocks the left-moving leg and `CTCF_RIGHT` the right-moving leg, so to hold
a loop between monomers `a < b` put `a` in `CTCF_LEFT` and `b` in `CTCF_RIGHT`.
`tile_sites(base, period, length)` repeats a pattern along the chain; for per-site
strengths pass a dict `{index: prob}` instead of a list (then `STALL` is ignored).

Times are in LEF timesteps throughout. Record for at least a few lifetimes:
`NUMSAVE * SAVEEVERY` should comfortably exceed `LIFE`.

In [ ]:
# --- chain and LEFs ---
NPOLY   = 70000        # monomers
SEP     = 240          # monomers per LEF -> NLEFS = NPOLY // SEP
LIFE    = 75000        # LEF lifetime, in LEF timesteps
VLEF    = 0.005        # p(step per leg per timestep)

# --- CTCF ---
STALL      = 0.8       # stall probability per encounter
CTCF_LEFT  = extrusion.tile_sites([200, 330, 724, 1425, 1433, 1604], period=2000, length=NPOLY)  # right pointing
CTCF_RIGHT = extrusion.tile_sites([574, 694, 866, 1241, 1390, 1580, 1752, 1800], period=2000, length=NPOLY)  # left pointing
LIFEBOOSTSTALLED = 4   # lifetime multiplier while stalled at a CTCF (1 = no boost)

# --- schedule ---
INITSTEPS = 2_000_000  # equilibration steps, discarded
NUMSAVE   = 36_000     # recorded frames
SAVEEVERY = 500        # LEF steps between frames

NLEFS = NPOLY // SEP
print("{0} LEFs, recording {1} frames over {2} LEF steps ({3:.1f} lifetimes)".format(
    NLEFS, NUMSAVE, NUMSAVE * SAVEEVERY, NUMSAVE * SAVEEVERY / LIFE))


291 LEFs, recording 36000 frames over 18000000 LEF steps (240.0 lifetimes)


## Build the LEF arrays

`build_lef_arrays` returns the six per-monomer arrays the translocator runs on. The
lifetime boost works by lowering `stallDeath` at the CTCF sites: a leg only ever sits
stalled where its stall probability is nonzero.

Note the translocator takes `max(falloff_left, falloff_right)`, so a LEF's lifetime is
only actually extended once *both* legs are stalled.

In [19]:
stall_left, stall_right = extrusion.build_stall_arrays(
    NPOLY, ctcf_left=CTCF_LEFT, ctcf_right=CTCF_RIGHT, stall_prob=STALL)

arrays = extrusion.build_lef_arrays(
    NPOLY, lifetime=LIFE, vlef=VLEF,
    stall_left=stall_left, stall_right=stall_right,
    life_boost_stalled=LIFEBOOSTSTALLED)

smc = extrusion.make_translocator(arrays, NLEFS)

n_sites = int(((stall_left > 0) | (stall_right > 0)).sum())
print("{0} CTCF sites; lifetime {1} while moving, {2:g} while stalled".format(
    n_sites, LIFE, LIFE * LIFEBOOSTSTALLED))

490 CTCF sites; lifetime 75000 while moving, 300000 while stalled


## Run

Equilibrate, then record leg positions every `SAVEEVERY` steps into an array of shape
`(NUMSAVE, NLEFS, 2)` — `[..., 0]` is the left leg, `[..., 1]` the right.

In [20]:
tstart = time.time()
smc.steps(INITSTEPS)

positions = np.zeros((NUMSAVE, NLEFS, 2), dtype=np.int64)
for i in tqdm(range(NUMSAVE)):
    smc.steps(SAVEEVERY)
    left, right = smc.getLEFs()
    positions[i, :, 0] = left
    positions[i, :, 1] = right

print("done in {0:.1f} s -> positions{1}".format(time.time() - tstart, positions.shape))

100%|██████████| 36000/36000 [01:03<00:00, 563.37it/s]

done in 71.0 s -> positions(36000, 291, 2)


## Save

In [ ]:
OUTDIR = "/mnt/md1/jjusuf/polysim/outputs/lef1d"
os.makedirs(OUTDIR, exist_ok=True)

np.save(os.path.join(OUTDIR, "LEFpositions.npy"), positions)
np.savez(os.path.join(OUTDIR, "sites.npz"), **arrays)
print("saved to", os.path.abspath(OUTDIR))

saved to /mnt/md0/jjusuf/polysim/polysim_bootcamp/main_example/outputs/lef1d


# Analyze

In [25]:
num_regions = 35
region_size = 2000
region_starts = np.arange(num_regions) * region_size

def in_contact(pos1, pos2, SMC_pos):
    if pos1 in SMC_pos and pos2 in SMC_pos:
        pos1_ind = np.where(SMC_pos==pos1)[0][0]
        pos2_ind = np.where(SMC_pos==pos2)[0][0]
        SMC_pos_ext = np.concatenate((SMC_pos, SMC_pos+1, SMC_pos-1), axis=1)
        N = SMC_pos_ext.shape[0]
        common_matrix = np.zeros((N, N), dtype=bool)
        for i in range(N):
            for j in range(N):
                if np.any(np.isin(SMC_pos_ext[i,:],SMC_pos_ext[j,:])):
                    common_matrix[i, j] = True
        for k in range(N):
            for i in range(N):
                for j in range(N):
                    common_matrix[i, j] = common_matrix[i, j] or (common_matrix[i, k] and common_matrix[k, j])
        return common_matrix[pos1_ind, pos2_ind]
    else:
        return False

def calculate_GT_loop_prob_CTCF(pos1, pos2, block_start, block_end, block_step):
    #print(f'progress:\n[{" "*num_regions}]')
    #print('[', end='')
    mean_contacts = np.zeros(num_regions)
    for region_num in range(num_regions):
        region_start = region_starts[region_num]
        region_end = region_start + region_size
        blocks_to_sample = range(block_start, block_end, block_step)
        contact_arr = np.zeros(len(blocks_to_sample), dtype='int')
        for i, block_num in enumerate(blocks_to_sample):
            SMC_pos_all = positions[block_num,:,:]
            SMC_pos = SMC_pos_all[np.all((np.all(SMC_pos_all>=region_start, 1), np.all(SMC_pos_all<region_end, 1)), 0),:]  # filter for SMCs on chain
            SMC_pos = SMC_pos - region_start  # offset SMC positions to be in interval [0, mapN)
            if in_contact(pos1, pos2, SMC_pos):
                contact_arr[i] = 1
        mean_contacts[region_num] = np.mean(contact_arr)
        #print(f'=', end='')
    #print(']')
    return np.mean(mean_contacts)


In [29]:
calculate_GT_loop_prob_CTCF(724, 866, 0, positions.shape[0], 100)

0.3633333333333333